# **MÓDULO 26 - Projeto Final do Aprofundamento de Analytics**

Bem-vindos ao Projeto de Dashboard de E-commerce! Este projeto é uma oportunidade para vocês aplicarem habilidades essenciais de análise de dados em um cenário prático e realista. Vocês irão trabalhar com um conjunto de dados de transações de clientes de uma loja virtual, distribuídos em duas tabelas distintas. O objetivo final é construir um dashboard interativo que facilite a visualização e análise das informações relevantes do e-commerce, utilizando ferramentas como Looker Studio ou Power BI.

**Objetivo do Projeto:**

Tratamento de Dados: Realizar a junção (JOIN) de duas tabelas utilizando SQL para consolidar as informações.
Análise de Dados: Exportar os dados resultantes para um arquivo CSV.
Visualização de Dados: Desenvolver um dashboard interativo e informativo para visualização das principais métricas e insights do e-commerce.

**Tabelas Disponibilizadas:**

**Tabela de Transações:** Contém os registros de transações realizadas pelos clientes, incluindo detalhes como ID da transação, valor e outros.


**Tabela de Dados Pessoais:** Contém as informações pessoais dos clientes, como ID do cliente, nome, genero, cidade, etc.

**Chave de Ligação:** As tabelas se relacionam através da coluna ID_CLIENT, que é a chave identificadora dos clientes.

# Etapas do Projeto:

1. Realizar um JOIN SQL nas duas tabelas, unificando as informações através da coluna ID_CLIENT. Você deve justificar a escolha do JOIN (Inner/ Left/ Right ou Full).

2. Exportar os dados consolidados resultantes do JOIN para um arquivo CSV.

3. Utilizar Looker Studio ou Power BI para importar o arquivo CSV.

4. Criar visualizações interativas que apresentem métricas importantes, como total de vendas, número de transações, distribuição geográfica dos clientes, perfil demográfico dos clientes, entre outros.

Abaixo temos a configuração do ambiente SQL:

In [1]:
import sqlite3
import pandas as pd

In [2]:
df_transacoes = pd.read_csv("TB_TRANSACOES_PROJETO_ECOMM.csv", delimiter=';')
df_clientes = pd.read_csv("TB_CLIENTES_PROJETO_ECOMM.csv", delimiter=';')

Antes de enviar as tabelas para o banco de dados, vale a pena dar uma primeira olhada na estrutura de cada uma, para entender quais colunas existem e como elas se relacionam.



In [ ]:
# Verificando as colunas e o tamanho de cada tabela

print("Tabela de transacoes:", df_transacoes.shape)
print(df_transacoes.columns.tolist())
print()
print("Tabela de clientes:", df_clientes.shape)
print(df_clientes.columns.tolist())

Tabela de transacoes: (367, 4)
['id_client', 'Category', 'Price', 'Card Type']

Tabela de clientes: (175, 5)
['state_name', 'First_name', 'Gender', 'Job_Title', 'Id_client']


A coluna Price da tabela de transações está no formato de texto, com vírgula como separador decimal. Essa conversão será feita mais adiante, já no arquivo final exportado, para não alterar a estrutura original das tabelas antes do JOIN em SQL.



In [ ]:
conn = sqlite3.connect('projeto.db')

# Carregar o DataFrame no banco de dados SQLite - criando tb_transacoes e tb_clientes

df_transacoes.to_sql('tb_transacoes', conn, index=False, if_exists='replace')
df_clientes.to_sql('tb_clientes', conn, index=False, if_exists='replace')

175

In [ ]:
# Função para executar consultas SQL e retornar o resultado como um DataFrame

def run_query(query):
    return pd.read_sql_query(query, conn)

# Etapa 1) Realizar um JOIN SQL nas duas tabelas, unificando as informações através da coluna ID_CLIENT.



In [6]:
query = '''
SELECT
    t.id_client,
    t.Category,
    t.Price,
    t."Card Type",
    c.state_name,
    c.First_name,
    c.Gender,
    c.Job_Title
FROM tb_transacoes t
INNER JOIN tb_clientes c
    ON t.id_client = c.Id_client
'''
result_df = run_query(query)
print(result_df)

     id_client     Category  ...       Gender                      Job_Title
0           37  Electronics  ...  Genderqueer                         Editor
1           38      Jewelry  ...         Male              Assistant Manager
2           39         Baby  ...       Female              Financial Analyst
3           40     Outdoors  ...       Female                 Civil Engineer
4            5     Outdoors  ...       Female                   VP Marketing
..         ...          ...  ...          ...                            ...
291        120         Baby  ...         Male    Computer Systems Analyst IV
292        121     Clothing  ...       Female                        Actuary
293        122     Clothing  ...       Female          Programmer Analyst II
294        123        Books  ...         Male  Analog Circuit Design manager
295        124       Health  ...       Female                   VP Marketing

[296 rows x 8 columns]


Justifique a escolha do JOIN:



Foi escolhido o INNER JOIN. A tabela de transações possui 367 registros, envolvendo 241 ids de cliente distintos, mas a tabela de clientes possui apenas 175 clientes cadastrados. Ao comparar as duas tabelas, identificou-se que existem 71 transações associadas a ids de cliente que não possuem nenhum cadastro na tabela de clientes, ou seja, transações sem nenhuma informação demográfica disponível, e 5 clientes cadastrados que nunca realizaram nenhuma transação. Como o objetivo principal do dashboard é justamente cruzar as transações com o perfil demográfico e geográfico dos clientes, um INNER JOIN garante que apenas as 296 transações com informação completa de cliente sejam levadas para o dashboard, evitando registros com estado, gênero ou cargo em branco, que poderiam distorcer os gráficos de distribuição geográfica e perfil demográfico. A desvantagem dessa escolha é que as transações órfãs ficam de fora da base final, então métricas de faturamento total ficariam um pouco subestimadas em relação ao total real, mas essa perda foi considerada aceitável em troca de uma base mais consistente para as análises demográficas que são o foco do projeto.

Exportando o arquivo como CSV:



In [ ]:
# Convertendo a coluna Price de texto para numero decimal, antes da exportação

result_df['Price'] = result_df['Price'].str.replace(',', '.', regex=False).astype(float)

result_df.to_csv('dados_ecommerce_final.csv', index=False)

result_df.head()

,id_client,Category,Price,Card Type,state_name,First_name,Gender,Job_Title
0,37,Electronics,72.93,mastercard,ND,Cornie,Genderqueer,Editor
1,38,Jewelry,121.89,mastercard,PA,Rab,Male,Assistant Manager
2,39,Baby,64.30,mastercard,MA,Codie,Female,Financial Analyst
3,40,Outdoors,9.48,mastercard,OR,Scott,Female,Civil Engineer
4,5,Outdoors,61.95,mastercard,MN,Tanney,Female,VP Marketing


**Dicas para o projeto:**
- Se atente que, como o mesmo cliente realiza mais de 1 transação quando você for trazer alguma métrica relacionada a dados do cliente terá que utilizar o distinct para criar essas métricas no dashboard, se não acabará tendo os dados repetidos.

- Análise sua tabela, entenda a dimensão dos dados, no excel, antes de enviar para o Powerbi ou Looker Studio.

- Tente montar preveamente um roteiro de quais métricas e visualizações irá colocar no dashboard, isso tornará seu processo mais rápido.

- Qualquer dificuldade para subir sua base para as ferramentas de visualização envie a base e o erro encontrado para que os tutores possam te ajudar.